# DL · Encuentro 2 · Imágenes, CNN y transferencia

De los píxeles a la representación

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [Tres preguntas distintas para una imagen](#scrollTo=tema-dl-vision)
2. [Aprender patrones locales de píxeles](#scrollTo=tema-dl-cnn)
3. [Reutilizar lo aprendido](#scrollTo=tema-dl-transferencia)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · 10 · Tres tipos de salida

**Lo que vas a poder hacer:** Definir la salida visual que necesita una operación y probar una detección.

Evita pedir posiciones o contornos a una herramienta que solamente clasifica.

### Entiende la idea

Una imagen digital está formada por pequeños cuadritos llamados píxeles. Cada uno guarda números que representan su brillo o color. Visión por computador es usar esos números para responder preguntas sobre imágenes. La primera decisión es qué respuesta necesitamos: una etiqueta para toda la foto, la ubicación de cada objeto o su superficie visible. Son tareas diferentes. Clasificación responde sobre la foto; detección propone objetos y ubicaciones; segmentación marca regiones de píxeles. Una máscara es el conjunto de píxeles marcados, no un rectángulo alrededor del objeto.

### Un caso pequeño antes del programa

**Una foto de tres frutos, tres necesidades**

Un centro de recepción recibe una foto ilustrada con tres frutos. El responsable puede querer seleccionar la foto para revisión, localizar cada fruto o examinar su región visible. Conservamos exactamente la misma escena y cambiamos la pregunta. Las salidas dibujadas permiten entender la tarea; no provienen de una red entrenada.

| Pregunta del equipo | Salida necesaria | Uso posible |
| --- | --- | --- |
| ¿Esta escena contiene frutos? | Una etiqueta para la escena | Seleccionar fotos |
| ¿Dónde está cada fruto? | Tres cajas de ubicación | Revisar posiciones |
| ¿Qué región visible ocupa cada uno? | Tres máscaras | Examinar contornos |

1. Para seleccionar la foto basta una etiqueta global; esa etiqueta no cuenta ni localiza cada objeto.
2. Para ubicar los tres frutos necesitamos cajas. Cada caja también contiene algo de fondo.
3. Para delimitar la región visible necesitamos máscaras, que distinguen píxeles del objeto y del entorno.

**Qué demuestra:** La salida que necesita el trabajo define la tarea; las tres no son intercambiables.

### Ahora identifica los datos de esta práctica

Compara etiqueta de imagen, cajas y regiones. Las marcas están dibujadas para explicar tareas; no las produjo un detector entrenado.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** En el cuaderno hay dos objetos ilustrados y en el aula, tres frutos. Compara el tipo de salida, no esperes que ambos dibujos tengan el mismo conteo.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### DL1: nombres que encontrarás al calcular y entrenar

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **DL** — DL: Deep Learning (Aprendizaje profundo). Parte del aprendizaje automático que usa redes neuronales con varias capas para transformar información y aprender representaciones. Una capa transforma números y entrega otros a la siguiente. Más capas no garantizan mejores resultados. [Fuente](https://cloud.google.com/discover/what-is-deep-learning)

- **ReLU** — ReLU: Rectified Linear Unit (Unidad lineal rectificada). Función de activación que devuelve el mayor entre la entrada y cero. Conserva números positivos y convierte los negativos en cero. Su salida no tiene límite superior y no representa por sí sola una probabilidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **BCE** — BCE: Binary Cross Entropy (Entropía cruzada binaria). Pérdida que compara una salida binaria probabilística con su respuesta conocida. En el caso sin pesos calcula −[y ln(p)+(1−y) ln(1−p)]. No tiene unidades físicas y no equivale a un porcentaje de errores; cuanto menor, mejor ajuste de esa pérdida sobre los mismos casos. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)

- **BCEWithLogitsLoss** — BCEWithLogitsLoss: Binary Cross Entropy with Logits Loss (Pérdida de entropía cruzada binaria a partir de logits). Función de PyTorch que recibe logits y combina internamente sigmoide con entropía cruzada binaria. Evita separar ambas operaciones para mejorar la estabilidad numérica. En el cuaderno no se añade una sigmoide antes de pasar la salida a esta pérdida. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

- **Adam** — Adam: Adaptive Moment Estimation (Estimación adaptativa de momentos). Optimizador que ajusta los parámetros usando promedios de los gradientes y de sus cuadrados. El nombre procede de esas estimaciones adaptativas. Tiene una tasa global y un ajuste dependiente del historial para cada parámetro; no aplica simplemente el mismo paso del descenso básico. [Fuente](https://arxiv.org/abs/1412.6980)


In [ ]:
# Ilustración exacta de tres TIPOS de salida, usando objetos sintéticos.
from matplotlib.patches import Rectangle
h,w=60,90;fila,col=np.mgrid[:h,:w]
mascara=((fila-30)**2+(col-25)**2<13**2)|((fila-28)**2+(col-65)**2<10**2)
imagen=np.ones((h,w,3))*.93;imagen[mascara]=[.2,.65,.4]
fig,axs=plt.subplots(1,3,figsize=(11,3))
for ax in axs:ax.axis('off')
axs[0].imshow(imagen);axs[0].set_title('Clasificación: «hay objetos»')
axs[1].imshow(imagen);axs[1].add_patch(Rectangle((12,17),26,26,fill=False,ec='red',lw=2));axs[1].add_patch(Rectangle((55,18),20,20,fill=False,ec='red',lw=2));axs[1].set_title('Detección: ¿dónde están?')
axs[2].imshow(mascara,cmap='Greens');axs[2].set_title('Segmentación: ¿qué píxeles?');mostrar('10-tareas-vision')
print('Estas cajas y máscaras se dibujaron a partir de la geometría: no ejecutan YOLO ni U-Net.')


### Después de ejecutar: ¿qué encargo requiere cada salida?

Para seleccionar imágenes basta una etiqueta global. Para señalar dónde está cada objeto hacen falta cajas. Para delimitar su región se usan píxeles marcados. Aquí hay dos objetos dibujados; el aula usa tres frutos. Cambia la escena, no la diferencia entre las tareas.


### DL3: secuencias y su error

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **RNN** — RNN: Recurrent Neural Network (Red neuronal recurrente). Red que procesa una secuencia manteniendo un resumen interno llamado estado. En cada paso combina la nueva entrada con ese estado usando los mismos parámetros aprendidos. El estado depende del orden; no es necesariamente el pronóstico de la próxima lectura. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)

- **LSTM** — LSTM: Long Short-Term Memory (Memoria de largo y corto plazo). Tipo de red recurrente con un estado de celda y operaciones llamadas compuertas. Estas regulan qué información se conserva, se incorpora y se transmite al procesar la secuencia. En esta práctica usa doce lecturas observadas de la serie simulada para pronosticar un paso; no anticipa sesenta pasos sin recibir nuevas observaciones. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

- **MAE** — MAE: Mean Absolute Error (Error absoluto medio). Promedio de las diferencias sin signo entre cada predicción y su respuesta observada: se resta, se elimina el signo y se promedian las distancias. Menor es mejor sobre los mismos casos y unidades; 0 indica coincidencia en todos esos casos. Conserva la unidad de la respuesta y no significa que cada caso tenga ese mismo error. Si se observan 4 y 10 cajas y se predicen 3 y 7, los errores absolutos son 1 y 3 cajas: MAE = (1+3)/2 = 2 cajas. [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)


### Lee, comprueba y explica

Compara etiqueta de imagen, cajas y regiones. Las marcas están dibujadas para explicar tareas; no las produjo un detector entrenado.

**Prueba pequeña:** En el cuaderno hay dos objetos ilustrados y en el aula, tres frutos. Compara el tipo de salida, no esperes que ambos dibujos tengan el mismo conteo.

**Vuelve al caso:** Pide una salida acorde con el trabajo: etiqueta para clasificar, cajas para localizar y máscaras para delimitar píxeles.

**Pregunta:** ¿Qué etiquetas hay que preparar para clasificar, detectar y segmentar?

**Respuesta razonada:** Para clasificación, una categoría por imagen en la tarea descrita; para detección, categoría y caja por objeto; para segmentación, categorías o identidades por píxel. Son anotaciones diferentes y requieren evaluaciones acordes.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-vision)


## Tema 2 · 10 · CNN y filtro

**Lo que vas a poder hacer:** Construir un mapa de filtro y distinguir cambios de imagen de cambios de pesos.

Ayuda a distinguir píxeles, pesos y resultados, antes de interpretar lo que reconoce una red.

### Entiende la idea

Para entender una CNN (Convolutional Neural Network; red neuronal convolucional), empieza con un cuadrito de imagen: un píxel guarda un número. Una ventana reúne unos pocos píxeles vecinos. Un filtro es una pequeña lista de pesos que se coloca sobre esa ventana: multiplica cada píxel por el peso correspondiente y suma. Después se mueve y repite la misma operación. Los números obtenidos forman el mapa de respuestas. Una CNN es una red que usa este tipo de operaciones locales y aprende filtros con ejemplos. Nuestro primer filtro está elegido de antemano para ver de dónde sale cada número; una respuesta del filtro aún no identifica un producto.

### Un caso pequeño antes del programa

**Encontrar un cambio entre filas**

La imagen inventada tiene 5 × 5 píxeles: dos filas de ceros y tres de unos (0 oscuro; 1 claro). La primera ventana de 3 × 3 contiene las filas de esta tabla. Usamos un filtro fijo para detectar el cambio. En este cálculo el paso es 1 y no añadimos relleno; por eso la salida tiene 3 × 3 celdas.

| Fila de píxeles | Pesos correspondientes | Suma de productos |
| --- | --- | --- |
| 0, 0, 0 | −1, −1, −1 | 0 |
| 0, 0, 0 | 0, 0, 0 | 0 |
| 1, 1, 1 | 1, 1, 1 | 3 |

1. Coloca el filtro sobre la primera ventana de 3 × 3. Multiplica cada píxel por el peso de la misma posición, sin invertir el filtro, como en una capa CNN.
2. Suma los nueve productos: 0 + 0 + 3 = 3. Esa es una celda del mapa.
3. Al bajar la ventana a la tercera fila, todos sus píxeles son 1: −3 + 0 + 3 = 0. Una región uniforme responde diferente.

**Qué demuestra:** Desplazar el mismo filtro produce nueve respuestas, un mapa de 3 × 3.

### Ahora identifica los datos de esta práctica

Distingue el filtro fijo que explica una operación de la CNN que aprende con ejemplos. Su tarea es reconocer dígitos, no clasificar frutos.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Una respuesta del filtro no es un objeto detectado. Relaciona cada celda del mapa con la zona de imagen que la produjo.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### DL1: nombres que encontrarás al calcular y entrenar

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **DL** — DL: Deep Learning (Aprendizaje profundo). Parte del aprendizaje automático que usa redes neuronales con varias capas para transformar información y aprender representaciones. Una capa transforma números y entrega otros a la siguiente. Más capas no garantizan mejores resultados. [Fuente](https://cloud.google.com/discover/what-is-deep-learning)

- **ReLU** — ReLU: Rectified Linear Unit (Unidad lineal rectificada). Función de activación que devuelve el mayor entre la entrada y cero. Conserva números positivos y convierte los negativos en cero. Su salida no tiene límite superior y no representa por sí sola una probabilidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

- **MSE** — MSE: Mean Squared Error (Error cuadrático medio). Promedio de las diferencias al cuadrado entre predicción y respuesta observada: cada diferencia se multiplica por sí misma antes de promediar. Menor es mejor y 0 indica coincidencia; sus unidades están al cuadrado. Los errores grandes pesan más que en el MAE, por lo que ambas métricas pueden preferir modelos diferentes. Con errores de 1 y 3 cajas: MSE = (1²+3²)/2 = (1+9)/2 = 5 cajas². [Fuente](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

- **BCE** — BCE: Binary Cross Entropy (Entropía cruzada binaria). Pérdida que compara una salida binaria probabilística con su respuesta conocida. En el caso sin pesos calcula −[y ln(p)+(1−y) ln(1−p)]. No tiene unidades físicas y no equivale a un porcentaje de errores; cuanto menor, mejor ajuste de esa pérdida sobre los mismos casos. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)

- **BCEWithLogitsLoss** — BCEWithLogitsLoss: Binary Cross Entropy with Logits Loss (Pérdida de entropía cruzada binaria a partir de logits). Función de PyTorch que recibe logits y combina internamente sigmoide con entropía cruzada binaria. Evita separar ambas operaciones para mejorar la estabilidad numérica. En el cuaderno no se añade una sigmoide antes de pasar la salida a esta pérdida. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)

- **Adam** — Adam: Adaptive Moment Estimation (Estimación adaptativa de momentos). Optimizador que ajusta los parámetros usando promedios de los gradientes y de sus cuadrados. El nombre procede de esas estimaciones adaptativas. Tiene una tasa global y un ajuste dependiente del historial para cada parámetro; no aplica simplemente el mismo paso del descenso básico. [Fuente](https://arxiv.org/abs/1412.6980)


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
D=load_digits();X=D.images.astype('float32')/16;y=D.target
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
xt=torch.tensor(Xtr[:,None]);yt=torch.tensor(ytr,dtype=torch.long);xe=torch.tensor(Xte[:,None])
torch.manual_seed(SEMILLA)
cnn=nn.Sequential(nn.Conv2d(1,8,kernel_size=3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(8*4*4,10))
opt=torch.optim.Adam(cnn.parameters(),lr=.01);perdida=nn.CrossEntropyLoss();lotes=DataLoader(TensorDataset(xt,yt),batch_size=64,shuffle=True)
EPOCAS=12;hist=[]
for epoca in range(EPOCAS):
    cnn.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(cnn(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(xt))
cnn.eval()
with torch.no_grad():pred=cnn(xe).argmax(1).numpy()
print('CNN · accuracy en prueba:',round(accuracy_score(yte,pred),3))
fig,axs=plt.subplots(2,6,figsize=(11,4))
for i,ax in enumerate(axs.ravel()):
    ax.imshow(Xte[i],cmap='gray');ax.set_title(f'Real {yte[i]} → pred. {pred[i]}',fontsize=9);ax.axis('off')
mostrar('10-cnn')
# Convolución con filtro fijo para visualizar la operación (no es un filtro aprendido).
filtro=torch.tensor([[[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]]])
with torch.no_grad():bordes=torch.nn.functional.conv2d(xe[:1],filtro,padding=1).numpy()[0,0]
fig,axs=plt.subplots(1,3,figsize=(10,3));axs[0].imshow(Xte[0],cmap='gray');axs[0].set_title('Entrada: 8×8 píxeles');axs[1].imshow(bordes,cmap='coolwarm');axs[1].set_title('Respuesta del filtro: cambios horizontales');axs[2].plot(hist,color=AZUL);axs[2].set(title='Error de la CNN',xlabel='Época',ylabel='Pérdida');mostrar('10-convolucion')


### Después de ejecutar: localiza un acierto y un error

Mira una imagen y compara la etiqueta Real con pred. Si no hay errores en las doce imágenes dibujadas, no concluyas que no los hay en toda la prueba: el dibujo solo muestra una parte. El mapa de borde corresponde a otro filtro fijo; sus colores no son clases de dígitos.


### 10B · Comparar etiqueta, cajas y regiones

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


### Lee, comprueba y explica

Distingue el filtro fijo que explica una operación de la CNN que aprende con ejemplos. Su tarea es reconocer dígitos, no clasificar frutos.

**Prueba pequeña:** Una respuesta del filtro no es un objeto detectado. Relaciona cada celda del mapa con la zona de imagen que la produjo.

**Vuelve al caso:** Cada celda del mapa sale de una ventana, los mismos pesos y una suma de productos. Un filtro fijo permite verificar esa operación.

**Pregunta:** ¿Un filtro que responde a un borde ya reconoce un fruto?

**Respuesta razonada:** No. Produce un mapa de respuestas locales. Una CNN combina muchos filtros, activaciones y capas hasta obtener una salida de la tarea; sus pesos se aprenden con datos. El filtro fijo permite entender una sola operación.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-cnn)


## Tema 3 · 2–6 · Reutilizar pesos y 9 · Filtrar categorías

**Lo que vas a poder hacer:** Probar si un modelo existente cubre una necesidad antes de adaptarlo.

Permite decidir si un modelo existente responde al problema o necesita adaptación.

### Entiende la idea

Un modelo preentrenado ya cambió sus pesos mientras aprendía con ejemplos anteriores. Usarlo para calcular una respuesta nueva se llama inferencia. Transferencia significa aprovechar parte de lo aprendido para otra tarea. Pero primero hay que comprobar qué categorías reconoce y en qué condiciones aprendió. Si necesitamos una categoría distinta, puede hacer falta entrenar de nuevo parte del modelo con ejemplos adecuados: eso es ajuste fino. Cambiar el texto de una etiqueta no cambia los pesos y no enseña una categoría nueva.

### Un caso pequeño antes del programa

**Reutilizar un detector para una tarea nueva**

Una cooperativa quiere detectar granos y distinguir su condición. Primero revisa las categorías del detector cargado: el archivo utilizado en el cuaderno reconoce objetos comunes y no incluye las categorías de café requeridas. La tabla distingue usar el modelo, traducir su interfaz y proponer un nuevo entrenamiento.

| Acción | ¿Actualiza pesos? | Qué significa |
| --- | --- | --- |
| Cargar pesos y predecir una foto | No | Reutiliza lo aprendido |
| Traducir los nombres de salida al español | No | Hace legible la interfaz; conserva las categorías aprendidas |
| Ajustar parámetros con café etiquetado | Sí | Podría adaptar el modelo; requiere evaluación |

1. Compara la categoría que necesitas con las categorías del archivo cargado.
2. Traducir una categoría conserva sus patrones y parámetros: mejora la lectura, pero no añade una categoría nueva.
3. Adaptar requiere ejemplos etiquetados, ajuste apropiado y evaluación reservada; el nombre del modelo no basta.

**Qué demuestra:** Cargar los pesos permite predecir las categorías disponibles. Para abordar café defectuoso hacen falta una tarea, datos etiquetados y una evaluación propios; cambiar el texto de una categoría mantiene el mismo detector.

### Ahora identifica los datos de esta práctica

Busca una categoría en el catálogo del modelo y luego en las propuestas de la foto. No estás entrenando una categoría nueva.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Filtrar «persona» conserva una clase existente. Renombrarla como «café» solo cambiaría el texto y sería una interpretación incorrecta.


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


In [ ]:
modelo = YOLO('yolo26n.pt')
categorias = pd.DataFrame([{'id':i,'clase_original':nombre,'nombre':nombre_es(nombre)}
                           for i,nombre in modelo.names.items()])
print('Categorías conocidas:',len(categorias))
print('Ejemplos del catálogo:',', '.join(categorias.nombre.head(15)))
print('Son solo ejemplos; una categoría no mostrada aún puede existir. La búsqueda consulta el catálogo completo.')
BUSCAR_CATEGORIA='café' # después prueba 'persona' sin cambiar los pesos
consulta=BUSCAR_CATEGORIA.strip().casefold()
coincidencias=categorias[(categorias.nombre.str.casefold().str.contains(consulta,regex=False))|
                         (categorias.clase_original.str.casefold().str.contains(consulta,regex=False))] if consulta else categorias.iloc[:0]
if coincidencias.empty:
    print('No hay coincidencias para',repr(BUSCAR_CATEGORIA),'en este catálogo. No renombres otra clase para fingir que la reconoce.')
else:
    print('Coincidencias en todo el catálogo:',', '.join(coincidencias.nombre))


In [ ]:
foto = ASSETS / 'bus.jpg'
original = Image.open(foto).convert('RGB')
print('Ancho y alto:',original.size,'· canales: RGB')
fig,ax=plt.subplots(figsize=(5,7))
ax.imshow(original);ax.axis('off');ax.set_title('Entrada: la foto sin predicciones')
plt.tight_layout();plt.savefig(SALIDA/'01_original.png',dpi=130);plt.show()


In [ ]:
UMBRAL = 0.25
resultado = modelo.predict(source=str(foto), conf=UMBRAL, imgsz=640,
                           device='cpu', verbose=False)[0]
print('Detecciones conservadas:',len(resultado.boxes))
resultado.save(filename=str(SALIDA/'02_foto_detectada.jpg'))
fig,axs=plt.subplots(1,2,figsize=(10,7))
axs[0].imshow(original);axs[0].set_title('Antes: píxeles')
axs[1].imshow(resultado.plot()[:,:,::-1]);axs[1].set_title('Después: categorías, cajas y puntuaciones')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.savefig(SALIDA/'03_antes_despues.png',dpi=130);plt.show()


In [ ]:
def tabla_detecciones(r):
    columnas=['deteccion','clase_id','clase_original','categoria','puntuacion',
              'x1','y1','x2','y2','ancho_px','alto_px']
    filas=[]
    for i,caja in enumerate(r.boxes, start=1):
        clase=int(caja.cls.item());nombre=r.names[clase]
        x1,y1,x2,y2=caja.xyxy[0].cpu().numpy().tolist()
        filas.append([i,clase,nombre,nombre_es(nombre),float(caja.conf.item()),
                      x1,y1,x2,y2,x2-x1,y2-y1])
    return pd.DataFrame(filas,columns=columnas)

tabla = tabla_detecciones(resultado)
display(tabla.round(3))
assert len(tabla)==len(resultado.boxes)
if not tabla.empty:
    primera=tabla.iloc[0]
    print('Primera detección:',primera.categoria,
          '· puntuación:',round(primera.puntuacion,3),
          '· tamaño:',round(primera.ancho_px),'×',round(primera.alto_px),'px')


In [ ]:
id_persona=next(i for i,n in modelo.names.items() if n=='person')
solo_personas=modelo.predict(str(foto),classes=[id_persona],conf=UMBRAL,
                             imgsz=640,device='cpu',verbose=False)[0]
display(tabla_detecciones(solo_personas).round(3))
if len(solo_personas.boxes):
    x1,y1,x2,y2=solo_personas.boxes.xyxy[0].cpu().numpy()
    recorte=original.crop((max(0,int(x1)),max(0,int(y1)),
                           min(original.width,int(x2)),min(original.height,int(y2))))
    fig,ax=plt.subplots(figsize=(3,4));ax.imshow(recorte);ax.axis('off')
    ax.set_title('Un recorte de la primera caja de persona');plt.tight_layout();plt.show()
else:print('No hay personas conservadas con este umbral.')


### Lee, comprueba y explica

Busca una categoría en el catálogo del modelo y luego en las propuestas de la foto. No estás entrenando una categoría nueva.

**Prueba pequeña:** Filtrar «persona» conserva una clase existente. Renombrarla como «café» solo cambiaría el texto y sería una interpretación incorrecta.

**Vuelve al caso:** Comprueba entradas y categorías del modelo. Reutilizar pesos, traducir nombres y ajustar parámetros son operaciones diferentes.

**Pregunta:** ¿Cargar un detector y traducir sus categorías constituye ajuste fino?

**Respuesta razonada:** No. Cargarlo e inferir conserva los parámetros; traducir cambia el texto visible. El ajuste fino requiere entrenamiento adicional con datos y una pérdida. Evaluar el dominio local sigue siendo necesario.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-transferencia)


## Cierra el encuentro con una explicación

1. **Tres preguntas distintas para una imagen:** Pide una salida acorde con el trabajo: etiqueta para clasificar, cajas para localizar y máscaras para delimitar píxeles.
2. **Aprender patrones locales de píxeles:** Cada celda del mapa sale de una ventana, los mismos pesos y una suma de productos. Un filtro fijo permite verificar esa operación.
3. **Reutilizar lo aprendido:** Comprueba entradas y categorías del modelo. Reutilizar pesos, traducir nombres y ajustar parámetros son operaciones diferentes.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
